<a href="https://colab.research.google.com/github/abdulrahman0700/intern_FlyRank_ai/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulrahman0700/intern_FlyRank_ai/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
"""
Flag content that is losing clicks quickly despite the underlying search demand still being there —
because that's a page that used to work and could work again, versus a page that never had traffic to lose.
"""

"\nFlag content that is losing clicks quickly despite the underlying search demand still being there — \nbecause that's a page that used to work and could work again, versus a page that never had traffic to lose.\n"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# --- Eligibility: exclude pages with too little prior traffic to judge a "drop" ---
# A page with 1 click going to 0 is a 100% drop but tells us almost nothing.
MIN_PRIOR_CLICKS = 3
eligible = df[df['clicks_prev_30d'] >= MIN_PRIOR_CLICKS].copy()

# --- Decline magnitude ---
# Only count actual declines (trend_pct is negative). Pages trending up/stable
# get a decline score of 0 -- they are not part of "losing clicks fast".
decline_pct = eligible['trend_pct'].clip(upper=0).abs()  # e.g. -41.4 -> 41.4, +12 -> 0
eligible['decline_score'] = (decline_pct / 100).clip(upper=1.0)  # 0-1, 1 = 100%+ drop

# --- Search volume weight ---
# search_volume is heavily right-skewed (median 10, max 22,200), so raw values
# would let a handful of huge-volume rows dominate. Log-scale, then normalize 0-1.
eligible['search_volume'] = eligible['search_volume'].fillna(0)
log_vol = np.log1p(eligible['search_volume'])
eligible['volume_score'] = (log_vol - log_vol.min()) / (log_vol.max() - log_vol.min())

# --- Combined baseline action score ---
# Multiplicative: a page needs BOTH real decline AND real demand to score high.
# A page with 0 search volume scores 0 no matter how fast it's declining,
# and a stable/growing page scores 0 no matter how much demand exists.
eligible['baseline_action_score'] = (eligible['decline_score'] * eligible['volume_score']).round(4)

# --- Reason codes ---
def reason_code(row):
    if row['decline_score'] >= 0.5 and row['volume_score'] >= 0.5:
        return "fast_decline_high_demand"
    elif row['decline_score'] >= 0.5:
        return "fast_decline_low_demand"
    elif row['volume_score'] >= 0.5:
        return "slow_decline_high_demand"
    else:
        return "minor_decline_low_demand"

eligible['reason_code'] = eligible.apply(reason_code, axis=1)

# --- Rank and write ---
ranked = eligible.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
ranked.insert(0, 'rank', ranked.index + 1)

output_cols = ['rank', 'content_id', 'client_id', 'baseline_action_score', 'reason_code',
               'decline_score', 'volume_score', 'trend_pct', 'search_volume',
               'clicks_last_30d', 'clicks_prev_30d', 'avg_position', 'position_tier',
               'main_intent', 'content_type', 'content_age_days', 'days_since_last_update']

ranked[output_cols].to_csv("baseline_action_score.csv", index=False)

print(f"Eligible pool: {len(eligible)} of {len(df)} total rows")
print(f"\nReason code distribution:")
print(ranked['reason_code'].value_counts())
print(f"\nTop 10:")
print(ranked[output_cols].head(10).to_string(index=False))

Eligible pool: 6705 of 30000 total rows

Reason code distribution:
reason_code
minor_decline_low_demand    4969
fast_decline_low_demand     1367
slow_decline_high_demand     301
fast_decline_high_demand      68
Name: count, dtype: int64

Top 10:
 rank           content_id         client_id  baseline_action_score              reason_code  decline_score  volume_score  trend_pct  search_volume  clicks_last_30d  clicks_prev_30d  avg_position position_tier   main_intent    content_type  content_age_days  days_since_last_update
    1 content_c861e30f2f7a client_6208ef0f77                 0.8820 fast_decline_high_demand          0.882      1.000000      -88.2        22200.0                0                4           8.4        page_1    commercial keyword article               229                     104
    2 content_548f0bf562a9 client_3fdba35f04                 0.7998 fast_decline_high_demand          0.870      0.919314      -87.0         9900.0                1                3         

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

ranked = pd.read_csv("/content/content_refresh_anonymized.csv")

top20 = ranked.head(20)
print(f"Top 20 of {len(ranked)} eligible pages\n")
print(top20.to_string(index=False))

# Use this table to fill in, for each of the 20 rows above:
#   - action (e.g. refresh / rewrite / consolidate / deprioritize)
#   - which reason_code applies and whether you agree with it
#   - a confidence note (what makes you trust or distrust this pick)
#   - what evidence would make this pick WRONG (write it before you'd know)


Top 20 of 30000 eligible pages

          content_id         client_id  search_volume  competition competition_level  cpc    content_type   main_intent  word_count  char_count provider_used             model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier   ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
content_304f48230142 client_f369cb89fc           10.0         0.67              HIGH 2.05 keyword article transactional      3221.0     20457.0           NaN       gemini-2.5-flash             3803          29             22            17         16               

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
ranked = pd.read_csv("/content/baseline_action_score.csv")
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# --- Check 1: does the score correlate suspiciously with fields that
#     shouldn't have driven it (e.g. anything derived from outcomes
#     after the decision point, or provider/model metadata)? ---
check_cols = ['content_age_days', 'days_since_last_update', 'word_count',
              'ai_traffic_pct', 'engagement_rate']
for col in check_cols:
    merged = ranked[['content_id', 'baseline_action_score']].merge(
        df[['content_id', col]], on='content_id', how='left')
    corr = merged['baseline_action_score'].corr(merged[col])
    print(f"corr(baseline_action_score, {col}): {corr:.3f}")

print()

# --- Check 2: timing sanity -- does last_30d + prev_30d overlap or gap
#     with anything else in the row that implies future knowledge? ---
window_check = df[['content_id', 'impressions_last_30d', 'impressions_prev_30d',
                    'impressions_90d']].copy()
window_check['sum_30d_windows'] = (window_check['impressions_last_30d'] +
                                     window_check['impressions_prev_30d'])
window_check['diff_from_90d'] = window_check['impressions_90d'] - window_check['sum_30d_windows']
print(window_check['diff_from_90d'].describe())
# If last_30d + prev_30d is consistently << impressions_90d, there's a
# 30-day gap unaccounted for -- check whether that gap could hide leakage
# or just reflects a >60-day 90d window (88-day trailing window, etc).

print()

# --- Check 3: any rows where trend_pct implies impossible values
#     (e.g. computed off a denominator of 0, or the sign disagrees
#     with trend_direction)? ---
sign_mismatch = df[
    ((df['trend_direction'] == 'up') & (df['trend_pct'] < 0)) |
    ((df['trend_direction'] == 'down') & (df['trend_pct'] > 0))
]
print(f"Rows where trend_direction and trend_pct sign disagree: {len(sign_mismatch)}")

# Use the output above to write your leakage verdict and list any weak
# picks from Section 3 that these checks help explain.


corr(baseline_action_score, content_age_days): 0.078
corr(baseline_action_score, days_since_last_update): -0.123
corr(baseline_action_score, word_count): -0.196
corr(baseline_action_score, ai_traffic_pct): -0.045
corr(baseline_action_score, engagement_rate): -0.000

count     30000.000000
mean       1988.229067
std        6099.411747
min           0.000000
25%          28.000000
50%         308.000000
75%        1512.250000
max      258505.000000
Name: diff_from_90d, dtype: float64

Rows where trend_direction and trend_pct sign disagree: 0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.